In [1]:
import os, sys

# Drop inherited PYTHONPATH inside this kernel
os.environ.pop("PYTHONPATH", None)

# Remove old Pinocchio/Raisim paths from sys.path if they slipped in
bad_substrings = [
    "Pinnochio_ws/local_install",
    "Pinnochio_Test/local_install",
    "raisim_legged/raisim_workspace/raisim/linux/lib",
    "matlab_legged/Dynamics_Generation/Pinnochio",
]

sys.path = [
    p for p in sys.path
    if not any(bad in (p or "") for bad in bad_substrings)
]

import pinocchio as pin
from pinocchio import casadi as cpin

print("pinocchio:", pin.__file__)
print("pinocchio.casadi:", cpin)


pinocchio: /home/trec/anaconda3/envs/Pinocchio/lib/python3.10/site-packages/pinocchio/__init__.py
pinocchio.casadi: <module 'pinocchio.casadi' from '/home/trec/anaconda3/envs/Pinocchio/lib/python3.10/site-packages/pinocchio/casadi/__init__.py'>


In [2]:
#!/usr/bin/env python3
import os
import numpy as np
import pinocchio as pin
import casadi
from pinocchio import casadi as cpin


In [3]:
# === CONFIGURATION ===

urdf_path   = "../../rsc/Go2/kinova_modified1_wofingers.urdf"  # Path to the URDF file
export_dir  = "./generated"



# 2) Load standard Pinocchio model & data
model = pin.buildModelFromUrdf(urdf_path, pin.JointModelFreeFlyer())
data  = model.createData()
print(f"[INFO] Loaded model with nq={model.nq}, nv={model.nv}")


# 3) Wrap into CasADi‐compatible Pinocchio model & data
cmodel = cpin.Model(model)
cdata  = cmodel.createData()
# q_trial = cmodel.randomConfiguration()

gen = casadi.CodeGenerator("kinova_cpin_dynamics.c")
(data.M).shape


[INFO] Loaded model with nq=11, nv=10


(10, 10)

In [4]:
for i, fr in enumerate(cmodel.frames):
    print(f"{i:3d}  {fr.name:25s}  {fr.type}")

  0  universe                   FIXED_JOINT
  1  root_joint                 JOINT
  2  j2n4s300_link_base         BODY
  3  j2n4s300_joint_1           JOINT
  4  j2n4s300_link_1            BODY
  5  j2n4s300_joint_2           JOINT
  6  j2n4s300_link_2            BODY
  7  j2n4s300_joint_3           JOINT
  8  j2n4s300_link_3            BODY
  9  j2n4s300_joint_4           JOINT
 10  j2n4s300_link_4            BODY
 11  j2n4s300_joint_end_effector  FIXED_JOINT
 12  j2n4s300_end_effector      BODY
 13  j2n4s300_joint_finger_1    FIXED_JOINT
 14  j2n4s300_link_finger_1     BODY
 15  j2n4s300_joint_finger_tip_1  FIXED_JOINT
 16  j2n4s300_link_finger_tip_1  BODY
 17  j2n4s300_joint_finger_2    FIXED_JOINT
 18  j2n4s300_link_finger_2     BODY
 19  j2n4s300_joint_finger_tip_2  FIXED_JOINT
 20  j2n4s300_link_finger_tip_2  BODY
 21  j2n4s300_joint_finger_3    FIXED_JOINT
 22  j2n4s300_link_finger_3     BODY
 23  j2n4s300_joint_finger_tip_3  FIXED_JOINT
 24  j2n4s300_link_finger_tip_3  BODY


In [5]:
def XYZEul2Quat(roll, pitch, yaw):
        """Convert XYZEuler angles to quaternion."""
        #Pinnochi has quat as x,y,z,w - opposite of raisim
        sroll = np.sin(roll / 2)
        croll = np.cos(roll / 2)
        spitch = np.sin(pitch / 2)
        cpitch = np.cos(pitch / 2)
        syaw = np.sin(yaw / 2)
        cyaw = np.cos(yaw / 2)
        quatw =  croll*cpitch*cyaw-sroll*spitch*syaw
        quatx =  sroll*cpitch*cyaw+croll*spitch*syaw
        quaty = -sroll*cpitch*syaw+croll*spitch*cyaw
        quatz =  croll*cpitch*syaw+sroll*spitch*cyaw
    
        return  quatx, quaty, quatz, quatw

In [6]:
def ZYXEul2Quat(roll, pitch, yaw):
        """Convert ZYX Euler angles to quaternion."""
        #Pinnochi has quat as x,y,z,w - opposite of raisim
        sroll = np.sin(roll / 2)
        croll = np.cos(roll / 2)
        spitch = np.sin(pitch / 2)
        cpitch = np.cos(pitch / 2)
        syaw = np.sin(yaw / 2)
        cyaw = np.cos(yaw / 2)
        quatw =  croll*cpitch*cyaw+sroll*spitch*syaw
        quatx =  sroll*cpitch*cyaw-croll*spitch*syaw
        quaty = croll*spitch*cyaw + sroll*cpitch*syaw
        quatz =  croll*cpitch*syaw-sroll*spitch*cyaw
    
        return  quatx, quaty, quatz, quatw

In [7]:
# 4) CasADi symbols for configuration & velocity
px = casadi.SX.sym("px")
py = casadi.SX.sym("py")
pz = casadi.SX.sym("pz")
# FreeFlyer has quaternion, but we need Euler angles
roll = casadi.SX.sym("roll")
pitch = casadi.SX.sym("pitch")
yaw = casadi.SX.sym("yaw")

quatx = casadi.SX.sym("quatx")
quaty = casadi.SX.sym("quaty")
quatz = casadi.SX.sym("quatz")
quatw = casadi.SX.sym("quatw")

q_joints = casadi.SX.sym("q_joints", model.nq - 7)  # Exclude FreeFlyer part
quatx, quaty, quatz, quatw = ZYXEul2Quat(roll, pitch, yaw)
q_eul = casadi.vertcat(px, py, pz, roll, pitch, yaw, q_joints)

q = casadi.vertcat(px, py, pz, quatx, quaty, quatz, quatw, q_joints)
v = casadi.SX.sym("v", model.nv)




In [ ]:

# 5) Dynamics

# 5a) Mass matrix D(q) via CRBA
cpin.crba(cmodel, cdata, q)
D_mat = casadi.Function("D_mat", [q_eul], [cdata.M], ["q_eul"], ["D_mat"])


gen.add(D_mat)
D_mat.save("D_mat_pin.casadi")

AttributeError: module 'pinocchio.casadi' has no attribute 'WORLD'

In [ ]:
# 5b) Coriolis+centrifugal vector C(q,v) via RNEA(q,v,0)
C_vec = casadi.Function(
    "C_vec",
    [q_eul, v],
    [ cpin.rnea(cmodel, cdata, q, v, casadi.SX.zeros(model.nv)) ],
    ["q_eul", "v"],
    ["C_vec"]
)
gen.add(C_vec)
#This is the Coriolis + gravity, but we don't use it for LL so I keep it like this

In [9]:
# 5c) Gravity vector G(q) via RNEA(q,0,0)
G_vec = casadi.Function(
    "G_vec",
    [q_eul],
    [ cpin.rnea(cmodel, cdata, q,
                casadi.SX.zeros(model.nv),
                casadi.SX.zeros(model.nv)) ],
    ["q_eul"],
    ["G_vec"]
)

gen.add(G_vec)
G_vec.save("G_vec_pin.casadi")

In [ ]:
# 5d) Actuation matrix B(q) = I
B_mat = casadi.Function("B_mat", [q_eul], [ np.eye(model.nv) ])
gen.add(B_mat)



In [ ]:
# 6) Kinematics & Jacobians & their time‐derivatives
toe_frames = ["FL_foot","FR_foot","RL_foot","RR_foot"]
hip_frames = ["FL_thigh_joint","FR_thigh_joint","RL_thigh_joint","RR_thigh_joint"]
all_frames = toe_frames + hip_frames

FK_funs = []
J_funs  = []
dJ_funs = []


In [ ]:
for name in all_frames:
    fid = cmodel.getFrameId(name)
    # --- FK: translation only
    cpin.framesForwardKinematics(cmodel, cdata, q)   # builds graph, returns None
    p = cdata.oMf[fid].translation                   # SX(3,1)
    FK_fun = casadi.Function(f"FK_{name}", [q_eul], [p])

    # --- Jacobian (linear part)
    J6 = cpin.computeFrameJacobian(cmodel, cdata, q, fid, pin.LOCAL_WORLD_ALIGNED)
    J_fun = casadi.Function(f"J_{name}", [q_eul], [J6[0:3, :]])

    # --- dJ (linear part)
    cpin.computeJointJacobiansTimeVariation(cmodel, cdata, q, v)  # None
    dJ6 = cpin.getFrameJacobianTimeVariation(cmodel, cdata, fid, pin.LOCAL_WORLD_ALIGNED)
    dJ_fun = casadi.Function(f"dJ_{name}", [q_eul, v], [dJ6[0:3, :]])


    FK_funs.append(FK_fun)
    J_funs.append(J_fun)
    dJ_funs.append(dJ_fun)



In [ ]:
for fun in FK_funs + J_funs + dJ_funs:
    gen.add(fun)

In [11]:
# 7) Export
gen.generate()

'kinova_cpin_dynamics.c'

In [ ]:
os.system("gcc -fPIC -shared -O3 kinova_cpin_dynamics.c -o kinova_cpin_dynamics.so")

In [ ]:
print(casadi.__version__)